# M2 Phase 2: DNS-BERT on the testbed (Experiments 1-4 + ablation 1 + per-host averaging)
Owner: Thierry · Backup: Yehan. Team Colab Pro: **Runtime → Change runtime type → T4 GPU**, then run the cells in order.

**One-time setup**
1. Google Drive → *Shared with me* → right-click **"leakmate captures"** → **Organize → Add shortcut → My Drive**.
2. Colab Secrets (key icon) → `GH_TOKEN` = a GitHub token with repo access, notebook access **on** (the repo is private).

**What it uses from "leakmate captures"**: the two testbed capture zips (`leakmate_captures_*.zip`), the CIC zips in `raw/cic/`,
and the Phase 1 models in `m2/ckpt_bpe_v2/last` and `m2/ckpt_char_v2/last`.

**What it writes there**: results in `m2/phase2_results/<model>/<run>.json` (`m2`, `m2_hostavg`, `m2_ablation1_char`,
`m2_ablation1_char_hostavg`, `iforest`, `iforest_hostavg`), trained models in `m2/phase2_weights/`.
If Colab disconnects, run the cells again: finished runs are skipped (`--skip-existing`).

Runs, for the BPE model and the character model (ablation 1), each also as `_hostavg` (per-host averaging), plus the Isolation Forest:
- **E2** CIC benchmark: new model on CIC benign + Tranco top 200k
- **E1** within-testbed: Phase 1 model + continued pre-training on testbed benign (benign only)
- **E3** leave-one-tool-out ×4: same model (M2 never trains on tunnels), test on each unseen tool
- **E4** cross-dataset: CIC→testbed (the E2 model) and testbed→CIC (new model, testbed benign only)

Threshold: 1% FPR on held-out benign (latest 10% of each capture), with the eval module's tie-aware `eval.metrics.threshold_at_fpr`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob
SHARED  = '/content/drive/MyDrive/leakmate captures'
M2_DIR  = f'{SHARED}/m2'
OUT     = f'{M2_DIR}/phase2_results'
WEIGHTS = f'{M2_DIR}/phase2_weights'
BRANCH  = 'main'
assert os.path.isdir(SHARED), ('"leakmate captures" not found in My Drive: Shared with me -> right-click it -> '
                               'Organize -> Add shortcut -> My Drive, then rerun this cell')
need = [f'{M2_DIR}/ckpt_bpe_v2/last/model.safetensors', f'{M2_DIR}/ckpt_char_v2/last/model.safetensors']
missing = [p for p in need if not os.path.exists(p)]
assert not missing, f'Phase 1 models missing: {missing}'
CAPTURE_ZIPS = sorted(glob.glob(f'{SHARED}/leakmate_captures_*.zip'))
print('capture zips:', [os.path.basename(z) for z in CAPTURE_ZIPS])
assert len(CAPTURE_ZIPS) >= 2, 'expected the iodine/dnscat2 and dns2tcp/DNSExfiltrator capture zips'
print('CIC zips:', sorted(os.listdir(f'{SHARED}/raw/cic')))
!mkdir -p "$OUT" "$WEIGHTS"

In [ ]:
%cd /content
# Private repo: GitHub token from Colab Secrets (key icon), named GH_TOKEN
from google.colab import userdata
try:
    tok = userdata.get('GH_TOKEN') + '@'
except Exception:
    tok = ''
!rm -rf leakmate && git clone -q -b $BRANCH https://{tok}github.com/TerminallyStoned/leakmate.git
%cd /content/leakmate
!git log --oneline -1
!pip -q install -r requirements-m2.txt

In [ ]:
# CIC table (Experiments 2 and 4) from the shared CIC zips, checked against the contract
import re, zipfile
def cic_target(name):
    n = re.sub(r'[^a-z]', '', os.path.basename(name).lower().replace('.zip', ''))
    for key, sub in [('lightattack', 'light_attacks'), ('heavyattack', 'heavy_attacks'),
                     ('lightbenign', 'light_benign'), ('heavybenign', 'heavy_benign'), ('benign', 'benign_main')]:
        if n.startswith(key):
            return sub
for z in glob.glob(f'{SHARED}/raw/cic/*.zip'):
    dest = f'data/raw/cic/{cic_target(z)}'
    zipfile.ZipFile(z).extractall(dest)
    for inner in glob.glob(f'{dest}/*.zip'):
        zipfile.ZipFile(inner).extractall(dest)
!python -m parser cic --raw data/raw/cic --out data/processed/cic/queries.parquet
!python contracts.py data/processed/cic/queries.parquet

In [ ]:
# Testbed captures: extract both zips (benign/, tools/); the table is built in the run cell.
# Tranco top list for Experiment 2 (downloaded once into the shared folder).
for z in CAPTURE_ZIPS:
    zipfile.ZipFile(z).extractall('data/raw/testbed')
!find data/raw/testbed -name '*.pcap' | wc -l
if not os.path.exists(f'{SHARED}/tranco.csv'):
    !wget -q -O /tmp/tranco.zip https://tranco-list.eu/top-1m.csv.zip && unzip -p /tmp/tranco.zip > "$SHARED/tranco.csv"
!mkdir -p data/external && cp "$SHARED/tranco.csv" data/external/tranco.csv

In [ ]:
# Full Phase 2: builds the testbed table (4h upstream-side benign run left out, team decision), then every M2 run
# (BPE + character ablation, each with _hostavg) and the Isolation Forest. About 45-75 min on a T4.
# Rerun this cell after a disconnect: finished runs are skipped.
!python scripts/m2_phase2.py --build-testbed data/raw/testbed \
    --phase1-bpe "$M2_DIR/ckpt_bpe_v2/last" --phase1-char "$M2_DIR/ckpt_char_v2/last" \
    --weights "$WEIGHTS" --out "$OUT" --skip-existing --iforest

In [ ]:
# OPTIONAL, only after a change to the scoring or threshold code: re-score every M2 run with the models already
# saved in phase2_weights (no training, ~25-35 min), then rerun the Isolation Forest. Skip it on a normal run.
!python scripts/m2_phase2.py --build-testbed data/raw/testbed --rescore --weights "$WEIGHTS" --out "$OUT"
!python scripts/m2_phase2.py --iforest-only --weights "$WEIGHTS" --out "$OUT"

In [ ]:
# Check every results file against the contract and print a quick summary, using the eval module's
# score >= threshold rule (official metrics: python -m eval.summarise)
!python contracts.py "$OUT"/*/*.json
import json, numpy as np
from sklearn.metrics import roc_auc_score
for f in sorted(glob.glob(f'{OUT}/*/*.json')):
    r = json.load(open(f)); y, s = np.array(r['labels']), np.array(r['scores'])
    rec = (s[y == 1] >= r['threshold']).mean(); fpr = (s[y == 0] >= r['threshold']).mean()
    print(f"{os.path.relpath(f, OUT):45s} AUC={roc_auc_score(y, s):.4f} recall={rec:.3f} FPR={fpr:.4f} "
          f"lat={r['latency_ms_per_query']:.2f}ms")

In [ ]:
# Download all Phase 2 results as one zip (a fresh zip each time)
!rm -f /content/m2_phase2_results.zip && cd "$OUT" && zip -qr /content/m2_phase2_results.zip .
from google.colab import files
files.download('/content/m2_phase2_results.zip')